# 5 · Visualisations

Generates all charts into `../charts/`.  
Cumulative returns use `(1 + r).cumprod() - 1` (compound), not `.cumsum()` (arithmetic).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D
from pathlib import Path
import sys
sys.path.insert(0, '..')
from utils import (
    ANALYSIS_DIR, CHARTS_DIR, RF_MONTHLY, RF_ANNUAL,
    PERIODS_PER_YEAR, apply_style
)

apply_style()

analysis_path = Path(ANALYSIS_DIR)
charts_path   = Path(CHARTS_DIR)
charts_path.mkdir(parents=True, exist_ok=True)

# ── Load data ─────────────────────────────────────────────────────────────────
returns = pd.read_csv(analysis_path / "india_equity_returns.csv",
                      index_col="Date", parse_dates=True)
prices  = pd.read_csv(analysis_path / "india_equity_prices.csv",
                      index_col="Date", parse_dates=True)
sim     = pd.read_csv(analysis_path / "portfolio_simulations.csv")
frontier= pd.read_csv(analysis_path / "efficient_frontier.csv")
metrics = pd.read_csv(analysis_path / "portfolio_metrics.csv")
weights = pd.read_csv(analysis_path / "portfolio_weights.csv", index_col="Asset")

corr      = returns.corr()
cov       = returns.cov()
mean_ret  = returns.mean()
vol       = returns.std()
n_assets  = len(returns.columns)

w_equal  = weights["Equal_Weight"].values
w_opt    = weights["MaxSharpe_Weight"].values
w_minvar = weights["MinVar_Weight"].values

max_sharpe_row = sim.loc[sim["Sharpe"].idxmax()]
min_vol_row    = sim.loc[sim["Volatility"].idxmin()]

# ─────────────────────────────────────────────────────────────────────────────
# 1. Correlation heatmap
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title(f"Asset Correlation Heatmap ({n_assets} Assets)")
fig.tight_layout()
fig.savefig(charts_path / "correlation_heatmap.png", dpi=150)
plt.close()

# ─────────────────────────────────────────────────────────────────────────────
# 2. Covariance heatmap
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cov * 100, annot=True, fmt=".3f", cmap="viridis",
            linewidths=0.5, ax=ax)
ax.set_title(f"Asset Covariance Matrix ({n_assets} Assets) — ×100")
fig.tight_layout()
fig.savefig(charts_path / "covariance_heatmap.png", dpi=150)
plt.close()

# ─────────────────────────────────────────────────────────────────────────────
# 3. Normalised price performance
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
(prices / prices.iloc[0]).plot(ax=ax)
ax.set_title(f"Normalised Price Performance ({n_assets} Stocks, base=1)")
ax.set_ylabel("Normalised Price")
ax.legend(loc="upper left", ncol=3)
fig.tight_layout()
fig.savefig(charts_path / "normalized_prices.png", dpi=150)
plt.close()

# ─────────────────────────────────────────────────────────────────────────────
# 4. Return distributions
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
for c in returns.columns:
    ax.hist(returns[c] * 100, bins=40, alpha=0.4, label=c)
ax.set_xlabel("Monthly Return (%)")
ax.set_title(f"Distribution of Monthly Returns ({n_assets} Stocks)")
ax.legend(ncol=3)
fig.tight_layout()
fig.savefig(charts_path / "return_distributions.png", dpi=150)
plt.close()

# ─────────────────────────────────────────────────────────────────────────────
# 5. Efficient frontier 2D
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 7))

sc = ax.scatter(
    sim["Volatility"] * 100, sim["Return"] * 100,
    c=sim["Sharpe"], cmap="viridis", s=5, alpha=0.6, linewidths=0
)
plt.colorbar(sc, ax=ax, label="Monthly Sharpe Ratio")

# True frontier line
ax.plot(
    frontier["Volatility"] * 100, frontier["Return"] * 100,
    color="black", linewidth=1.5, linestyle="-", label="Efficient Frontier"
)

# Key portfolios
ax.scatter(max_sharpe_row["Volatility"] * 100, max_sharpe_row["Return"] * 100,
           color="red", s=180, marker="*", zorder=5,
           label=f"Max Sharpe ({max_sharpe_row['Sharpe']:.3f})")
ax.scatter(min_vol_row["Volatility"] * 100, min_vol_row["Return"] * 100,
           color="blue", s=120, marker="D", zorder=5,
           label=f"Min Variance ({min_vol_row['Volatility']*100:.2f}% vol)")

# Capital Market Line
x_cml = np.linspace(0, sim["Volatility"].max() * 100, 200)
slope  = (max_sharpe_row["Return"] - RF_MONTHLY) / max_sharpe_row["Volatility"]
y_cml  = RF_MONTHLY * 100 + slope * x_cml
ax.plot(x_cml, y_cml, linestyle="--", color="darkorange",
        linewidth=1.2, label="Capital Market Line")

ax.set_xlabel("Monthly Volatility (%)")
ax.set_ylabel("Monthly Return (%)")
ax.set_title(
    f"Efficient Frontier  |  RF={RF_ANNUAL*100:.1f}% p.a.  "
    f"|  Max Sharpe={max_sharpe_row['Sharpe']:.3f}"
)
ax.legend(loc="upper left")
fig.tight_layout()
fig.savefig(charts_path / "efficient_frontier_2d.png", dpi=150)
plt.close()

# ─────────────────────────────────────────────────────────────────────────────
# 6. Efficient frontier 3D
# ─────────────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(11, 7))
ax3 = fig.add_subplot(111, projection="3d")

p = ax3.scatter(
    sim["Volatility"] * 100, sim["Return"] * 100, sim["Sharpe"],
    c=sim["Sharpe"], cmap="viridis", s=3, alpha=0.5
)
fig.colorbar(p, ax=ax3, label="Monthly Sharpe")

ax3.scatter(max_sharpe_row["Volatility"]*100, max_sharpe_row["Return"]*100,
            max_sharpe_row["Sharpe"], color="red", s=150, marker="*")
ax3.scatter(min_vol_row["Volatility"]*100, min_vol_row["Return"]*100,
            min_vol_row["Sharpe"], color="blue", s=100, marker="D")

ax3.set_xlabel("Volatility (%)")
ax3.set_ylabel("Return (%)")
ax3.set_zlabel("Sharpe")
ax3.set_title(f"3D Portfolio Risk–Return–Sharpe | Max Sharpe {max_sharpe_row['Sharpe']:.3f}")
fig.tight_layout()
fig.savefig(charts_path / "efficient_frontier_3d.png", dpi=150)
plt.close()

# ─────────────────────────────────────────────────────────────────────────────
# 7. Mean returns bar
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
(mean_ret * 100).plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.axhline(0, color="black", linewidth=0.8)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_xlabel("Ticker")
ax.set_ylabel("Mean Monthly Return (%)")
ax.set_title(f"Average Monthly Asset Returns ({n_assets} Stocks)")
plt.xticks(rotation=45)
fig.tight_layout()
fig.savefig(charts_path / "mean_returns_bar.png", dpi=150)
plt.close()

# ─────────────────────────────────────────────────────────────────────────────
# 8. Portfolio metrics comparison
# ─────────────────────────────────────────────────────────────────────────────
plot_cols = ["Sharpe_Annual", "Beta_Proxy", "Treynor_Monthly", "Jensen_Alpha_Mo"]
m_plot = metrics.set_index("Portfolio")[plot_cols]

fig, ax = plt.subplots(figsize=(11, 6))
m_plot.plot(kind="bar", ax=ax, edgecolor="white")
best = metrics.loc[metrics["Sharpe_Annual"].idxmax()]
ax.set_title(f"Portfolio Metrics | Best Annual Sharpe: {best['Sharpe_Annual']:.3f} ({best['Portfolio']})")
plt.xticks(rotation=0)
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(charts_path / "portfolio_metrics.png", dpi=150)
plt.close()

# ─────────────────────────────────────────────────────────────────────────────
# 9. Asset risk–return scatter
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(vol * 100, mean_ret * 100, s=80, zorder=3)
for ticker in mean_ret.index:
    ax.annotate(ticker,
                (vol[ticker] * 100, mean_ret[ticker] * 100),
                textcoords="offset points", xytext=(6, 4), fontsize=9)
ax.axhline(RF_MONTHLY * 100, color="grey", linestyle="--",
           linewidth=0.9, label=f"RF ({RF_MONTHLY*100:.3f}% / mo)")
ax.set_xlabel("Monthly Volatility (%)")
ax.set_ylabel("Mean Monthly Return (%)")
ax.set_title(f"Asset Risk–Return Profile ({n_assets} Assets)")
ax.legend()
fig.tight_layout()
fig.savefig(charts_path / "asset_risk_return_scatter.png", dpi=150)
plt.close()

# ─────────────────────────────────────────────────────────────────────────────
# 10. Cumulative portfolio returns  ← compound, not cumsum
# ─────────────────────────────────────────────────────────────────────────────
cum_equal  = ((1 + returns @ w_equal ).cumprod() - 1) * 100
cum_opt    = ((1 + returns @ w_opt   ).cumprod() - 1) * 100
cum_minvar = ((1 + returns @ w_minvar).cumprod() - 1) * 100

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(cum_equal.index,  cum_equal,  label="Equal Weight")
ax.plot(cum_opt.index,    cum_opt,    label="Max Sharpe")
ax.plot(cum_minvar.index, cum_minvar, label="Min Variance", linestyle="--")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_ylabel("Cumulative Return (%)")
ax.set_title(
    f"Portfolio Cumulative Returns (compound)  |  "
    f"Max Sharpe Monthly SR={max_sharpe_row['Sharpe']:.3f}"
)
ax.legend()
fig.tight_layout()
fig.savefig(charts_path / "portfolio_cumulative_returns.png", dpi=150)
plt.close()

# ─────────────────────────────────────────────────────────────────────────────
# 11. Portfolio weights
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))
(weights * 100).plot(kind="bar", ax=ax, edgecolor="white")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_xlabel("Stock")
ax.set_ylabel("Weight (%)")
ax.set_title(f"Portfolio Weight Allocation ({n_assets} Assets)")
plt.xticks(rotation=45)
fig.tight_layout()
fig.savefig(charts_path / "portfolio_weights.png", dpi=150)
plt.close()

# ─────────────────────────────────────────────────────────────────────────────
# 12. Sharpe comparison
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
metrics.set_index("Portfolio")["Sharpe_Annual"].plot(
    kind="bar", ax=ax, color=["#4e79a7", "#f28e2b", "#59a14f"], edgecolor="white"
)
best = metrics.loc[metrics["Sharpe_Annual"].idxmax()]
ax.set_title(f"Annual Sharpe Ratio Comparison | Best: {best['Sharpe_Annual']:.3f} ({best['Portfolio']})")
ax.set_ylabel("Annual Sharpe Ratio")
ax.axhline(0, color="black", linewidth=0.7)
plt.xticks(rotation=0)
fig.tight_layout()
fig.savefig(charts_path / "sharpe_comparison.png", dpi=150)
plt.close()

# ─────────────────────────────────────────────────────────────────────────────
# 13. Portfolio construction % (nb 7 equivalent)
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
(weights * 100).plot(kind="bar", ax=ax, edgecolor="white")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_title("Portfolio Construction (% Allocation)")
ax.set_ylabel("Weight (%)")
ax.set_xlabel("Stock")
plt.xticks(rotation=45)
fig.tight_layout()
fig.savefig(charts_path / "portfolio_construction_percent.png", dpi=150)
plt.close()

print(f"All charts saved to {charts_path.resolve()}")
print(f"Total charts generated: 13")

FileNotFoundError: [Errno 2] No such file or directory: '../analysis/portfolio_metrics.csv'